In [37]:
!pip install -q requests beautifulsoup4 tqdm


In [38]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.auto import tqdm

In [39]:
# List of URLs (sample blogs and Wikipedia pages)
urls = [
    "https://hairlust.com/blogs/blog",
    "https://www.pai-shau.com/blogs/hair-and-there",
    "https://theearthcollective.in/blogs/blog/creating-a-nourishing-hair-care-routine-your-guide-to-a-healthy-hair-scalp",
    "https://minimalistbeauty.com/blog/no-more-chemical-hair-care-part-1",
    "https://thefoundationblog.com/blog/naturalhaircare",
    "https://www.hairholistic.ca/blogs/blog",
    "https://bloggers.feedspot.com/hair_care_blogs/",
    "https://en.wikipedia.org/wiki/Hair_care",
    "https://en.wikipedia.org/wiki/Hair",
    "https://en.wikipedia.org/wiki/Hair_removal",
    "https://en.wikipedia.org/wiki/Plucking_(hair_removal)[11]",
    "https://huggingface.co/datasets/Amod/hair_medical_sit",
    "https://universe.roboflow.com/project-5mecq/hair-loss-x5sxe",
    "https://data.mendeley.com/datasets/g46n66frrh",
    "https://www.kaggle.com/datasets/trainingdatapro/hair-detection-and-segmentation-dataset",
    "https://www.ai-trichologist.com",
    "https://perfecthairhealth.com/free-resources/",
    "https://ihairium.com/web-app"
]

In [40]:
def scrape_text(url):
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")
        paragraphs = soup.find_all(['p', 'h1', 'h2', 'h3'])
        full_text = "\n".join(p.get_text(strip=True) for p in paragraphs)
        return full_text
    except Exception as e:
        print(f"Failed to scrape {url}: {e}")
        return ""

corpus = []
for url in tqdm(urls):
    text = scrape_text(url)
    if text:
        corpus.append({"url": url, "text": text})

df = pd.DataFrame(corpus)
df.to_csv("perp_haircare_blog_corpus.csv", index=False)
print("Scraping complete! Data saved to haircare_blog_corpus.csv")

  0%|          | 0/18 [00:00<?, ?it/s]

Scraping complete! Data saved to haircare_blog_corpus.csv


In [41]:
df

,url,text
0,https://hairlust.com/blogs/blog,Featured\nShop by Category\nShop by Attribute\...
1,https://www.pai-shau.com/blogs/hair-and-there,"Haircare Blog\nRecent\nGlossy, Frizz-Free, and..."
2,https://theearthcollective.in/blogs/blog/creat...,Diwali Sale Live. Flat 25% off Sitewide\nExtra...
3,https://minimalistbeauty.com/blog/no-more-chem...,No More Chemical Hair Care - Part 1\nI can't b...
4,https://thefoundationblog.com/blog/naturalhair...,My Personal Clean(ish) Haircare Routine\nLast ...
5,https://www.hairholistic.ca/blogs/blog,Earn points with each order!\nShop Scalpcare →...
6,https://bloggers.feedspot.com/hair_care_blogs/,Select Page\n70 Best Hair Care Blogs and Websi...
7,https://huggingface.co/datasets/Amod/hair_medi...,Datasets:Amod/hair_medical_sitlike2\nDataset S...
8,https://data.mendeley.com/datasets/g46n66frrh,\nDataset for Evaluating Hair Hall Causes Usin...
9,https://www.ai-trichologist.com,403\nForbidden


In [42]:

# Display the first few rows for inspection
print(df.head())

# Show basic info about the dataset
print("Number of rows:", len(df))
print("Columns:", df.columns.tolist())

# Optionally, display some sample text data (first 500 characters from first entry)
print(df['text'][0][:500])

                                                 url  \
0                    https://hairlust.com/blogs/blog   
1      https://www.pai-shau.com/blogs/hair-and-there   
2  https://theearthcollective.in/blogs/blog/creat...   
3  https://minimalistbeauty.com/blog/no-more-chem...   
4  https://thefoundationblog.com/blog/naturalhair...   

                                                text  
0  Featured\nShop by Category\nShop by Attribute\...  
1  Haircare Blog\nRecent\nGlossy, Frizz-Free, and...  
2  Diwali Sale Live. Flat 25% off Sitewide\nExtra...  
3  No More Chemical Hair Care - Part 1\nI can't b...  
4  My Personal Clean(ish) Haircare Routine\nLast ...  
Number of rows: 12
Columns: ['url', 'text']
Featured
Shop by Category
Shop by Attribute
Collections
Ingredients
The A to Z of our product ingredients.
Reviews
Before-after images and testimonials
Hair Talk Blog
A dictionary of hair care
Your Hair. Your Story.
It’s time to start the conversation, to share the untold tales of hair st

## Block 3: Clean and Chunk the Corpus

### Step 1: Remove Boilerplate and Non-Informative Text
Many scraped texts include headers, footers, navigation bars, or promotional material.
Use keyword filtering and basic text cleaning to keep relevant content.

In [43]:
import pandas as pd
import re

df = pd.read_csv("perp_haircare_blog_corpus.csv")

# Define function to remove boilerplate/noise (expand keywords as needed)
def clean_text(text):
    # Remove typical navigation/header/footer phrases
    boilerplate_patterns = [
        r"Featured", r"Shop by Category", r"Shop by Attribute",
        r"Collections", r"Ingredients", r"Reviews", r"Shop",
        r"Popular searches", r"Your search yielded no results",
        r"Diwali Sale Live", r"Last updated", r"Add Codeadd Markdown"
    ]
    for pat in boilerplate_patterns:
        text = re.sub(pat, '', text, flags=re.IGNORECASE)
    # Remove excessive newlines and spaces
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    return text.strip()

df['cleaned_text'] = df['text'].apply(clean_text)


### Step 2: Chunk Cleaned Texts Into Passages
Large blog posts and Wikipedia extracts work best when split into smaller passages for retrieval.

In [44]:
def chunk_text(text, chunk_size=400):
    sentences = re.split(r'(?<=[.!?]) +', text)   # Split at sentence boundaries
    chunks = []
    current = ""
    for sent in sentences:
        if len(current) + len(sent) < chunk_size:
            current += sent + " "
        else:
            chunks.append(current.strip())
            current = sent + " "
    if current.strip():
        chunks.append(current.strip())
    return chunks

chunked_data = []
for idx, row in df.iterrows():
    chunks = chunk_text(row['cleaned_text'])
    for chunk in chunks:
        if len(chunk) > 50:  # filter out tiny fragments
            chunked_data.append({'url': row['url'], 'text': chunk})

chunked_df = pd.DataFrame(chunked_data)
chunked_df.to_csv("haircare_chunks.csv", index=False)
print("Chunking complete. Data saved to haircare_chunks.csv")


Chunking complete. Data saved to haircare_chunks.csv


In [45]:
chuk = pd.read_csv("haircare_chunks.csv")
chuk

,url,text
0,https://hairlust.com/blogs/blog,The A to Z of our product .\nBefore-after imag...
1,https://hairlust.com/blogs/blog,We share our knowledge and make it easier for ...
2,https://hairlust.com/blogs/blog,7 Effective Ways to Fight Back Hair Loss Natur...
3,https://hairlust.com/blogs/blog,Mousse: How to Choose The Best Curly Hair Prod...
4,https://hairlust.com/blogs/blog,Best Antifungal Shampoos + Remedies for Scalp ...
...,...,...
269,https://perfecthairhealth.com/free-resources/,"These cost thousands of dollars, are supported..."
270,https://perfecthairhealth.com/free-resources/,We built and provide this service for free\nSa...
271,https://perfecthairhealth.com/free-resources/,"We do this for you, for free\nKnow the “real” ..."
272,https://perfecthairhealth.com/free-resources/,Comparisons are hard.\nSolution\nWe built an a...


# Block 4: Computing Embeddings and Building a Vector Store
Now you’ll encode each text passage into a high-dimensional vector, index those with FAISS, and prepare them for fast semantic search.

In [46]:
!pip install -q sentence-transformers faiss-cpu


In [47]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load chunked corpus
df = pd.read_csv("haircare_chunks.csv")

# Choose an efficient embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Compute sentence embeddings for all text chunks
embeddings = embedder.encode(df['text'].tolist(), show_progress_bar=True)

# Convert to numpy float32 array for FAISS
embedding_matrix = np.array(embeddings).astype('float32')

# Create and populate FAISS index
dimension = embedding_matrix.shape[1]  # Embedding dimension (usually 384)
index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

# Save the index and metadata for future use
faiss.write_index(index, "haircare_faiss.index")
df.to_csv("haircare_chunks_with_index.csv", index=False)
print("FAISS index and corpus saved! Ready for fast retrieval.")


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

FAISS index and corpus saved! Ready for fast retrieval.


In [48]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [49]:
!pip install -q sentence-transformers faiss-cpu

import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load the chunked corpus
df = pd.read_csv('haircare_chunks.csv')

# Use a fast, effective embedding model
model_name = 'all-MiniLM-L6-v2'
embedder = SentenceTransformer(model_name)

# Create embeddings for all text chunks
corpus = df['text'].tolist()
corpus_embeddings = embedder.encode(corpus, show_progress_bar=True)

# FAISS expects float32 numpy arrays
corpus_embeddings = np.array(corpus_embeddings).astype('float32')

# Build FAISS index with a unique name
haircare_faiss_index = faiss.IndexFlatL2(corpus_embeddings.shape[1])
haircare_faiss_index.add(corpus_embeddings)

# Save index and metadata for retrieval
faiss.write_index(haircare_faiss_index, 'haircare_faiss_custom.index')
df.to_csv('haircare_chunks_with_index.csv', index=False)
print("Embeddings and FAISS index saved as haircare_faiss_custom.index for fast retrieval!")


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embeddings and FAISS index saved as haircare_faiss_custom.index for fast retrieval!


# Block 5: Retrieval + Answer Generation Pipeline

In [50]:
!pip install -q transformers sentence-transformers faiss-cpu

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd

# Load FAISS index and corpus metadata
faiss_index = faiss.read_index("haircare_faiss_custom.index")
corpus_df = pd.read_csv("haircare_chunks_with_index.csv")

# Load embedding model for query encoding
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Load generation model and tokenizer (T5-small works well for Kaggle GPU)
tokenizer = AutoTokenizer.from_pretrained("t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small").to('cuda' if torch.cuda.is_available() else 'cpu')

def retrieve_top_k(query, k=3):
    query_embedding = embedder.encode([query])
    query_embedding = np.array(query_embedding).astype('float32')
    distances, indices = faiss_index.search(query_embedding, k)
    return [corpus_df.iloc[i]['text'] for i in indices[0]]

def generate_answer(query, max_length=150):
    retrieved_chunks = retrieve_top_k(query, k=3)
    context = " ".join(retrieved_chunks)
    input_text = f"question: {query} context: {context}"
    inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True).to(model.device)
    #outputs = model.generate(inputs, max_length=max_length, num_beams=3, early_stopping=True)
    #outputs = model.generate(inputs, max_length=250, num_beams=5, early_stopping=True)
    outputs = model.generate(
    inputs, 
    max_length=250, 
    num_beams=5, 
    temperature=0.7, 
    top_p=0.9, 
    early_stopping=True
)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

In [51]:
# Example usage
user_query = "What are some natural remedies for hair loss?"
print("User Query:", user_query)
print("Bot Answer:", generate_answer(user_query))

User Query: What are some natural remedies for hair loss?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Bot Answer: Natural and nourishing can work wonders for your hair health


# Fine tuning the mdoel on our corpus for better and accurate results


## Step 1: Improve Generation Output Quality

## Script to Generate Questions From Text Chunks Using a Small LLM

In [53]:
!pip install -q transformers

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
from tqdm.auto import tqdm

# Load your chunked corpus CSV
df = pd.read_csv("haircare_chunks.csv")

# Small GPT-2 model for question generation (faster & smaller)
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to('cuda' if torch.cuda.is_available() else 'cpu')

# Prompt template for question generation
def prepare_prompt(answer_text):
    prompt = f"Based on the following hair care information, generate a relevant question:\nAnswer: {answer_text}\nQuestion:"
    return prompt

# Function to generate question
def generate_question(text, max_length=64):
    prompt = prepare_prompt(text)
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
    inputs,
    max_new_tokens=64,          # Generate up to 64 new tokens after input
    num_beams=5,
    no_repeat_ngram_size=2,
    early_stopping=True
)

    question = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract text after 'Question:' for cleaner output
    if "Question:" in question:
        question = question.split("Question:")[-1].strip()
    return question

# Generate Q&A
qa_pairs = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    answer_text = row['text']
    question_text = generate_question(answer_text)
    # Filter short or non-questions
    if len(question_text) > 5 and question_text.endswith("?"):
        qa_pairs.append({"question": question_text, "answer": answer_text})

# Save generated Q&A
qa_df = pd.DataFrame(qa_pairs)
qa_df.to_csv("generated_haircare_qa_pairs.csv", index=False)
print(f"Generated {len(qa_df)} Q&A pairs saved to generated_haircare_qa_pairs.csv")


  0%|          | 0/274 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generati

Generated 87 Q&A pairs saved to generated_haircare_qa_pairs.csv


In [54]:
qna = pd.read_csv("generated_haircare_qa_pairs.csv")
qna.head()

,question,answer
0,What do you think of your hair products? Do yo...,The A to Z of our product .\nBefore-after imag...
1,What do you do with your body? What are you do...,And while flowers and...\nThe Ultimate Spring ...
2,What do you think is the most important color ...,Fall can be a tricky time for...\nEssential Fa...
3,What do you think of the beauty and beauty of ...,"From Hollywood icons to royalty, these legenda..."
4,What do you think is the most important ingred...,Look for products that contain:\nArgan Oil:Ric...


## better filtering for the qna pair and new model for the question making

In [55]:
!pip install -q transformers

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
from tqdm.auto import tqdm

# Load your chunked corpus
df = pd.read_csv("haircare_chunks.csv")

# Use small GPT model for question generation
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to('cuda' if torch.cuda.is_available() else 'cpu')

def prepare_prompt(answer_text):
    return f"Based on the following hair care information, generate a relevant question:\nAnswer: {answer_text}\nQuestion:"

def generate_questions(text, num_candidates=3, max_new_tokens=64):
    prompt = prepare_prompt(text)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    questions = []
    for seed in range(num_candidates):
        # Re-seed for diversity
        torch.manual_seed(seed)
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,              # Enable sampling for diversity
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            num_return_sequences=1,
            no_repeat_ngram_size=2,
            early_stopping=True
        )
        question = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "Question:" in question:
            question = question.split("Question:")[-1].strip()
        questions.append(question)
    return questions

# Generate multiple Q&A candidates per chunk
qa_candidates = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    answer_text = row['text']
    generated_questions = generate_questions(answer_text)
    for q in generated_questions:
        # Relaxed filtering for manual curation later
        if len(q) > 10:
            qa_candidates.append({"question": q, "answer": answer_text})

# Save all candidates to CSV for review
qa_df = pd.DataFrame(qa_candidates)
qa_df.to_csv("expanded_generated_haircare_qa_pairs.csv", index=False)
print(f"Generated {len(qa_df)} candidate Q&A pairs saved to expanded_generated_haircare_qa_pairs.csv")



  0%|          | 0/274 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask and the pad token id were not set. As a consequence, you may observe

Generated 822 candidate Q&A pairs saved to expanded_generated_haircare_qa_pairs.csv


In [56]:
chuckk = pd.read_csv("expanded_generated_haircare_qa_pairs.csv")
chuckk

,question,answer
0,How do you get your hair done?\nWe‭re asking y...,The A to Z of our product .\nBefore-after imag...
1,What is your favorite hair conditioner?\nYou m...,The A to Z of our product .\nBefore-after imag...
2,Does the hair look like a normal hair?\nWhat d...,The A to Z of our product .\nBefore-after imag...
3,How to Make Hair Hair for Women: What is Hair?...,We share our knowledge and make it easier for ...
4,What is Hair Stain? What Does It Do? For Hair ...,We share our knowledge and make it easier for ...
...,...,...
817,What is your favorite hair conditioner for you...,Comparisons are hard.\nSolution\nWe built an a...
818,Does the hump“hush effect in hair removal re...,Comparisons are hard.\nSolution\nWe built an a...
819,How is the free AI diagnostic work? How does t...,hairline check aitrichologist online consultat...
820,How can I find out about my hair? How does it ...,hairline check aitrichologist online consultat...


# Block 6: Fine-Tuning T5 on Generated Q&A Pairs

In [1]:
import transformers
print(transformers.__version__)


4.52.4
new


In [1]:
import time
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments, TrainerCallback

# Confirm GPU presence
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(torch.cuda.current_device()))

# Load dataset
df = pd.read_csv("/kaggle/input/expanded-hair-qna-pairs/expanded_generated_haircare_qa_pairs.csv")
dataset = Dataset.from_pandas(df.reset_index(drop=True))
split_ds = dataset.train_test_split(test_size=0.10, seed=42)
train_dataset = split_ds["train"].select(range(min(1000, len(split_ds["train"]))))
val_dataset = split_ds["test"].select(range(min(200, len(split_ds["test"]))))

# Load tokenizer and model; fix pad_token
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Preprocessing
def preprocess_function(examples):
    inputs = examples['question']
    targets = examples['answer']
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=256, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print(f"[{time.strftime('%H:%M:%S')}] Starting tokenization with multiprocessing...")
train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=["question", "answer"], num_proc=4)
val_dataset = val_dataset.map(preprocess_function, batched=True, remove_columns=["question", "answer"], num_proc=4)
print(f"[{time.strftime('%H:%M:%S')}] Tokenization complete.")

# Training args
training_args = TrainingArguments(
    output_dir="./t5_haircare_finetuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=50,
    report_to=[],
    fp16=False,
)

# Callback for logging
class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        print(f"Step {state.global_step}: {logs}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    callbacks=[PrintLossCallback]
)

print(f"[{time.strftime('%H:%M:%S')}] Starting training...")
trainer.train()
print(f"[{time.strftime('%H:%M:%S')}] Training completed.")


2025-10-10 21:12:52.659207: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760130772.849879      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760130772.916278      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


CUDA Available: True
Device Name: Tesla P100-PCIE-16GB


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

...

Map (num_proc=4):   0%|          | 0/83 [00:00<?, ? examples/s]

[21:13:14] Tokenization complete.
[21:13:14] Starting training...


/tmp/ipykernel_36/2166953826.py:60: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
50,1.999900
100,1.756300
150,1.670800
200,1.633600
250,1.636300


Step 50: {'loss': 1.9999, 'grad_norm': 0.5483126640319824, 'learning_rate': 4.121863799283154e-05, 'epoch': 0.5376344086021505}
Step 100: {'loss': 1.7563, 'grad_norm': 0.46899527311325073, 'learning_rate': 3.2258064516129034e-05, 'epoch': 1.075268817204301}
Step 150: {'loss': 1.6708, 'grad_norm': 0.6077788472175598, 'learning_rate': 2.3297491039426525e-05, 'epoch': 1.6129032258064515}
Step 200: {'loss': 1.6336, 'grad_norm': 1.6464548110961914, 'learning_rate': 1.4336917562724014e-05, 'epoch': 2.150537634408602}
Step 250: {'loss': 1.6363, 'grad_norm': 0.8713316917419434, 'learning_rate': 5.376344086021506e-06, 'epoch': 2.688172043010753}
Step 279: {'train_runtime': 42.4231, 'train_samples_per_second': 52.259, 'train_steps_per_second': 6.577, 'total_flos': 75013193465856.0, 'train_loss': 1.7204164347768258, 'epoch': 3.0}
[21:13:57] Training completed.


In [2]:
save_dir = "./t5_haircare_finetuned_model"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Model and tokenizer saved to {save_dir}")


Model and tokenizer saved to ./t5_haircare_finetuned_model


# Build Inference and Chatbot Pipeline
Create a function for generating answers from user queries using your fine-tuned model:

# # Step 2: Retrieval-Augmented Generation using your fine-tuned T5 model and FAISS index

# # Load FAISS index and corpus metadata

In [39]:
#load the corpus and the faiss indices
faiss_index = faiss.read_index("/kaggle/input/hair-faiss-custom-index/haircare_faiss_custom.index")
corpus_df = pd.read_csv("/kaggle/input/hair-chunks-w-index/haircare_chunks_with_index.csv")

In [3]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 62.9 MB/s eta 0:00:00:00:0100:01


# git version:


# new prompt:tea

In [43]:
import faiss
import pandas as pd
import torch
import re
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load pre-trained Flan-T5 for reliable RAG
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Retrieval setup
embedder = SentenceTransformer('all-MiniLM-L6-v2')
#faiss_index = faiss.read_index("haircare_faiss_custom.index")
#corpus_df = pd.read_csv("haircare_chunks_with_index.csv")

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'^[-•*]\s*', '', text, flags=re.MULTILINE)
    sentences = [s.strip() for s in text.split('.') if len(s.strip())>10]
    seen = set(); unique=[]
    for s in sentences:
        if s not in seen:
            unique.append(s); seen.add(s)
    return '. '.join(unique)

def retrieve_context(query, k=5):
    emb = embedder.encode([query]).astype('float32')
    dists, idxs = faiss_index.search(emb, k*2)
    candidates=[]
    for dist, idx in zip(dists[0], idxs[0]):
        txt = clean_text(corpus_df.iloc[idx]['text'])
        if txt.count('.')>=2 and 100<len(txt)<500:
            candidates.append((txt, dist))
    candidates.sort(key=lambda x: x[1])
    selected = [c[0] for c in candidates[:k]]
    combined = '. '.join(selected)
    return combined[:1000]+'...' if len(combined)>1000 else combined

def generate_answer_new(query):
    context = retrieve_context(query, k=4)
    if len(context)<50:
        return "Insufficient context to answer."
    prompt = f"""You are a product knowledge assistant.

Given the user's problem and the retrieved context, identify the single most relevant product or ingredient that directly solves the problem.

If the context lists multiple items, choose the one that best matches the user's need.
Explain it a bit why this product should be their first choice. Do not include extra words.
Return the name of the product or ingredient. NO Yapping.

Context:
{context}

User Query:
{query}

Answer:
"""
    inputs = tokenizer(prompt, return_tensors="pt",
                       max_length=768, truncation=True, padding=True).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            min_length=80,
            num_beams=8,
            do_sample=False,
            repetition_penalty=1.2,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("Answer in two paragraphs:")[-1].strip()

# Example
print(generate_answer_new("What products help with hair loss?"))


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Coconut Oil:Penetrates hair shafts, reducing protein loss and enhancing shine. Shea Butter:Provides intense moisture, making hair soft and manageable. Aloe Vera:Soothes the scalp, promotes hair growth, and reduces dandruff. Keratin:Repairs and strengthens damaged hair, increasing its resilience


In [44]:
queries = [
    "What causes hair loss in women?",
    "Best products for dry and damaged hair?", 
    "How to prevent dandruff naturally?",
    "What vitamins are good for hair growth?",
    "How often should I wash oily hair?",
    "Natural remedies for hair thinning?",
    "What ingredients should I avoid in shampoo?",
    "How to repair split ends without cutting?",
    "Best hair masks for curly hair?",
    "What foods promote healthy hair growth?"
]

for query in queries:
    print(query)
    print("Answer:", generate_answer_new(query))
    print("#" * 50)

What causes hair loss in women?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: a certain number of hairs every single day; however, if you are losing a lot of hair, and that too from a particular part of your scalp; then that is a problem. Hair Buddha BlogAbout- Minaz is a neuro-physiotherapist turned natural hair therapist. Her tested methods for multiple hair care issues have benefited many around the world.
##################################################
Best products for dry and damaged hair?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: Argan Oil: Rich in antioxidants and fatty acids, argan oil deeply moisturises and tames frizz. Coconut Oil:Penetrates hair shafts, reducing protein loss and enhancing shine. Shea Butter:Provides intense moisture, making hair soft and manageable. Aloe Vera:Soothes the scalp, promotes hair growth, a...
##################################################
How to prevent dandruff naturally?


...

Answer: Micropigmentation is the ultimate non-surgical solution to thinning hair, receding hairlines, alopecia, scar camouflage and male pattern baldness. Our goal is to transform your mirror moments with personalized hair care as the conduit. MOREFacebook54. 2KTwitter18. 2KInstagram761. 5KDomain Authority40Get Email Contact 39. Skalp Blog Blog
##################################################
Best hair masks for curly hair?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: curling straightener, collagen, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband, headband
##################################################
What foods promote healthy hair growth?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: healthy hair starts from the ROOT. By focusing on the scalp, I have noticed such a difference in my hair. They have all grown so fast and so strong since regularly incorporating collagen into my diet. I mix it into my morning coffee/matcha every morning, and that seems to do the trick!. So easy and has such an impact on hair health...it is absolutely crazy what your body can do when you give it the right t...
##################################################


# savin the model

In [45]:
save_dir = "./prompted_flan_t5_haircare_model"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Model and tokenizer saved to {save_dir}")


Model and tokenizer saved to ./prompted_flan_t5_haircare_model


# loading and using the model

In [38]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

load_dir = "./prompted_flan_t5_haircare_model"
tokenizer = AutoTokenizer.from_pretrained(load_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(load_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):